# DiffusionPen, generatie en evaluatie van Nederlandse adresregels

Verkennende notebook voor het Prime Vision afstudeerproject.

**Doel van deze notebook**
1. Laad de pretrained DiffusionPen-IAM checkpoint die in deze repo staat.
2. Genereer een paar Nederlandse adresregels (naam / straat / postcode + plaats) met een paar verschillende IAM writer stijlen.
3. Evalueer de gegenereerde samples op drie assen:
   - **Visuele inspectie**, hoe plausibel zien ze eruit?
   - **Leesbaarheid**, wat leest een off-the-shelf OCR engine eraf?
   - **Realisme tegenover echte data**, side-by-side met de echte samples in `AddressBlock examples/`.

**Kanttekeningen**
- DiffusionPen is getraind op **IAM** (Engels handschrift). Het reproduceert Nederlandse glyphs (a-z, 0-9, komma, punt, koppelteken, spatie) zonder problemen omdat die allemaal overlappen met de IAM character set, maar de writer stijlen zijn *Engelse handschrift* stijlen. Behandel dit als een baseline test, niet als een eindoordeel.
- De few-shot **style-image branch** (`img_feat=True`) heeft IAM word crops in `./iam_data/words` nodig die we lokaal niet hebben. We zetten daarom `img_feat=False` en conditioneren de style alleen op het writer-class label (één van de 339 IAM writers). Later kunnen we hier echte Prime Vision style references aan koppelen.
- Bij de eerste run downloadt `diffusers` stable-diffusion-v1-5 (VAE + DDIM scheduler) en HuggingFace downloadt `google/canine-c`. Zet eerst `HF_HOME` als je de cache ergens specifieks wil hebben.

## 1 · Imports en path setup

In [ ]:
import os, sys, json, random, copy, math
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from torch.nn import DataParallel
from PIL import Image
import matplotlib.pyplot as plt

# Zorg dat we train.py / unet.py kunnen importeren uit de DiffusionPen folder,
# ongeacht of de notebook geopend is vanuit notebooks/, de project root of DiffusionPen/.
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / 'train.py').exists() and (_base / 'unet.py').exists():
        DIFFUSIONPEN_DIR = _base
        break
    if (_base / 'DiffusionPen' / 'train.py').exists():
        DIFFUSIONPEN_DIR = _base / 'DiffusionPen'
        break
else:
    raise FileNotFoundError(
        f'Kon DiffusionPen/train.py niet vinden vanuit {Path.cwd()}. '
        'Open deze notebook vanuit de project root, de notebooks/ map of de DiffusionPen folder.'
    )
sys.path.insert(0, str(DIFFUSIONPEN_DIR))
os.chdir(DIFFUSIONPEN_DIR)
print('Working dir:', os.getcwd())

# train.py importeert wandb op module niveau (wordt alleen achter een flag gebruikt).
# Stub het zodat we wandb niet hoeven te pip-installen alleen om de helpers te gebruiken.
# diffusers probet wandb ook via importlib.util.find_spec, dus __spec__ moet gezet zijn.
try:
    import wandb
except ImportError:
    import types as _types
    import importlib.machinery as _machinery
    _w = _types.ModuleType('wandb')
    _w.__spec__ = _machinery.ModuleSpec('wandb', loader=None)
    _w.__version__ = '0.0.0-stub'
    _w.init = lambda *a, **k: None
    _w.log = lambda *a, **k: None
    _w.Image = lambda *a, **k: None
    _w.config = _types.SimpleNamespace(update=lambda *a, **k: None)
    sys.modules['wandb'] = _w

from unet import UNetModel
from feature_extractor import ImageEncoder
from train import Diffusion, EMA, crop_whitespace_width  # noqa: E402
from diffusers import AutoencoderKL, DDIMScheduler
from transformers import CanineModel, CanineTokenizer

print('torch', torch.__version__, '| CUDA beschikbaar:', torch.cuda.is_available())

## 2 · Config en seeds

`args` doet de argparse namespace na die `train.py` opbouwt, zodat we `Diffusion.sampling` op precies dezelfde manier kunnen aanroepen.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

args = SimpleNamespace(
    model_name='diffusionpen',
    level='word',
    img_size=(64, 256),
    channels=4,
    emb_dim=320,
    num_heads=4,
    num_res_blocks=1,
    save_path='./diffusionpen_iam_model_path',
    device=device,
    color=True,
    latent=True,
    img_feat=False,  # geen IAM word crops lokaal, dus style komt alleen van het writer label
    interpolation=False,
    dataparallel=False,
    mix_rate=None,
    style_path='./style_models/iam_style_diffusionpen.pth',
    stable_dif_path='stable-diffusion-v1-5/stable-diffusion-v1-5',  # HF hub id, valt terug op lokale dir als die bestaat
    unet='unet_latent',
)

# Als er een lokale stable-diffusion-v1-5 folder is, gebruik die (offline-vriendelijk).
local_sd = DIFFUSIONPEN_DIR / 'stable-diffusion-v1-5'
if local_sd.exists():
    args.stable_dif_path = str(local_sd)
print('SD path:', args.stable_dif_path)
print('Device :', args.device)

## 3 · Laad VAE, scheduler en text encoder

In [ ]:
device_ids = [0] if torch.cuda.is_available() else None

vae = AutoencoderKL.from_pretrained(args.stable_dif_path, subfolder='vae')
if device_ids is not None:
    vae = DataParallel(vae, device_ids=device_ids).to(args.device)
else:
    vae = DataParallel(vae).to(args.device)
vae.requires_grad_(False)

ddim = DDIMScheduler.from_pretrained(args.stable_dif_path, subfolder='scheduler')

tokenizer = CanineTokenizer.from_pretrained('google/canine-c')
text_encoder = CanineModel.from_pretrained('google/canine-c')
if device_ids is not None:
    text_encoder = DataParallel(text_encoder, device_ids=device_ids).to(args.device)
else:
    text_encoder = DataParallel(text_encoder).to(args.device)
text_encoder.eval()
print('VAE, DDIM en text encoder klaar.')

## 4 · Laad de UNet (denoiser) en EMA checkpoint

In [ ]:
STYLE_CLASSES = 339  # IAM
VOCAB_SIZE = 80      # len(character_classes) in train.py

unet = UNetModel(
    image_size=args.img_size,
    in_channels=args.channels,
    model_channels=args.emb_dim,
    out_channels=args.channels,
    num_res_blocks=args.num_res_blocks,
    attention_resolutions=(1, 1),
    channel_mult=(1, 1),
    num_heads=args.num_heads,
    num_classes=STYLE_CLASSES,
    context_dim=args.emb_dim,
    vocab_size=VOCAB_SIZE,
    text_encoder=text_encoder,
    args=args,
)
if device_ids is not None:
    unet = DataParallel(unet, device_ids=device_ids).to(args.device)
else:
    unet = DataParallel(unet).to(args.device)

ckpt_path = Path(args.save_path) / 'models' / 'ckpt.pt'
ema_path  = Path(args.save_path) / 'models' / 'ema_ckpt.pt'

unet.load_state_dict(torch.load(ckpt_path, map_location=args.device))
unet.eval()

ema = EMA(0.995)
ema_model = copy.deepcopy(unet).eval().requires_grad_(False)
ema_model.load_state_dict(torch.load(ema_path, map_location=args.device))
ema_model.eval()
print('UNet en EMA geladen vanuit', ckpt_path)

## 5 · Laad de style feature extractor

Ook met `img_feat=False` bouwen we dezelfde `feature_extractor`, zodat we deze later kunnen aanzetten wanneer we Prime Vision reference crops erop aansluiten.

In [ ]:
feature_extractor = ImageEncoder(model_name='mobilenetv2_100', num_classes=0, pretrained=True, trainable=True)
state = torch.load(args.style_path, map_location=args.device)
fe_state = feature_extractor.state_dict()
state = {k: v for k, v in state.items() if k in fe_state and fe_state[k].shape == v.shape}
fe_state.update(state)
feature_extractor.load_state_dict(fe_state)
if device_ids is not None:
    feature_extractor = DataParallel(feature_extractor, device_ids=device_ids).to(args.device)
else:
    feature_extractor = DataParallel(feature_extractor).to(args.device)
feature_extractor.requires_grad_(False)
feature_extractor.eval()
print('Style encoder geladen.')

## 6 · Diffusion sampler en woord-naar-image helper

`generate_word` is een dunne wrapper rond `Diffusion.sampling` die een strak gecropte grayscale PIL afbeelding voor één woord teruggeeft.

In [ ]:
from torchvision import transforms as T
import torchvision
import cv2

diffusion = Diffusion(img_size=args.img_size, args=args)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

def width_crop(img: Image.Image) -> Image.Image:
    """Crop alleen horizontale whitespace, behoud de originele hoogte en baseline.

    De originele `crop_whitespace_width` cropt eigenlijk beide assen, wat de
    relatieve lettergroottes verpest (korte woorden worden uitgerekt om bij
    lange woorden te passen). Wij willen het natuurlijke 64 px verticale frame
    van het model behouden.
    """
    arr = np.array(img)
    _, th = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(th)
    if coords is None:
        return img
    x, _, w, _ = cv2.boundingRect(coords)
    return img.crop((x, 0, x + w, img.height))

# Raw-output cache: (word, writer_id) -> grayscale PIL Image voor de width_crop.
# Wordt gevuld bij de eerste call, en laat re-layout experimenten de diffusion overslaan.
# Deze cel opnieuw runnen wist de cache. Roep _word_cache.clear() aan om handmatig te resetten.
_word_cache: dict = {}

@torch.no_grad()
def generate_word(word: str, writer_id: int, crop: bool = True) -> Image.Image:
    """Genereer één handgeschreven woord op 64x(<=256). Alleen width-crop, baseline blijft staan."""
    key = (word, writer_id)
    if key not in _word_cache:
        labels = torch.tensor([writer_id], dtype=torch.long, device=args.device)
        sampled = diffusion.sampling(
            ema_model, vae, n=len(labels), x_text=word, labels=labels, args=args,
            style_extractor=feature_extractor, noise_scheduler=ddim, transform=transform,
            character_classes=None, tokenizer=tokenizer, text_encoder=text_encoder, run_idx=None,
        )
        _word_cache[key] = torchvision.transforms.ToPILImage()(sampled.squeeze(0)).convert('L')
    img = _word_cache[key]
    if crop:
        img = width_crop(img)
    return img

# Smoke test: render één enkel token.
_demo = generate_word('Haag', writer_id=12)
plt.imshow(_demo, cmap='gray'); plt.axis('off'); plt.title('smoke test: "Haag", writer 12'); plt.show()


## 7 · Stel woorden samen tot een adresregel, en regels tot een blok

In [ ]:
# Layout knoppen. Background-clamp + min-blend + jitter + paper noise = geen zichtbare naden.
LINE_H = 64
BASELINE_JITTER = 3            # +/- verticale wiebel per woord, in pixels
GAP_MIN, GAP_MAX = 12, 22      # horizontale gap tussen woorden
LINE_GAP_MIN, LINE_GAP_MAX = 6, 14
BG = 255
BG_CLAMP = 220                 # elke pixel >= deze waarde wordt puur 255 voor het compositen
PAPER_NOISE = 2                # finale ±N gray-level ruis over het canvas
DIGIT_GAP = 2                  # strakke gap (px) tussen cijfers binnen een nummer token

def _clean_bg(arr: np.ndarray, threshold: int = BG_CLAMP) -> np.ndarray:
    """Duw bijna-witte pixels naar puur 255.

    Elke word image komt uit het model met een licht grijze achtergrond (gemiddeld ~220).
    Zonder deze stap laten aangrenzende word patches nog steeds vage rechthoekige randen
    zien, zelfs met min-blending, omdat hun achtergronden niet bit-identiek zijn.
    """
    out = arr.copy()
    out[out >= threshold] = 255
    return out

def _blend_paste(canvas: np.ndarray, patch: np.ndarray, x: int, y: int) -> None:
    """Composite `patch` op `canvas` op positie (x, y) met min-blending."""
    ph, pw = patch.shape
    H, W = canvas.shape
    y0, y1 = max(0, y), min(H, y + ph)
    x0, x1 = max(0, x), min(W, x + pw)
    if y0 >= y1 or x0 >= x1:
        return
    py0, py1 = y0 - y, y1 - y
    px0, px1 = x0 - x, x1 - x
    canvas[y0:y1, x0:x1] = np.minimum(canvas[y0:y1, x0:x1], patch[py0:py1, px0:px1])

def _add_paper_noise(arr: np.ndarray, rng, amplitude: int = PAPER_NOISE) -> np.ndarray:
    """Voeg een kleine uniforme ruisfloor toe over het hele canvas.

    Zodra de achtergrond niet meer perfect wit is, wordt elke resterende patch
    grens weggespoeld in de ruis. Het lijkt dan op papier textuur.
    """
    if amplitude <= 0:
        return arr
    noise = rng.integers(-amplitude, amplitude + 1, size=arr.shape, dtype=np.int16)
    return np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def _render_token(token: str, writer_id: int) -> np.ndarray:
    """Geef een schone achtergrond (H=64, W) uint8 array terug voor één adres token.

    Tokens met meerdere cijfers ("12", "2511", "27", etc.) worden cijfer-voor-cijfer
    gerenderd en gestitcht met DIGIT_GAP pixels tussen elk cijfer.

    Rationale: DiffusionPen is getraind op IAM Engelse woorden, die nauwelijks
    sequenties van meerdere cijfers bevatten. Bij conditioneren op "27" verspreidt
    het model de twee cijfers met letter-level spacing (~15 px), waardoor OCR ze
    als twee aparte tokens leest. Losse cijfers ("2", "7") zijn wel in distributie
    als standalone karakters en renderen schoon. Door ze met een 2 px gap te
    stitchen krijgen we visueel coherente nummers zonder het artefact.
    """
    if token.isdigit() and len(token) > 1:
        imgs = [_clean_bg(np.array(generate_word(d, writer_id))) for d in token]
        h = max(im.shape[0] for im in imgs)
        total_w = sum(im.shape[1] for im in imgs) + DIGIT_GAP * (len(imgs) - 1)
        canvas = np.full((h, max(1, total_w)), BG, dtype=np.uint8)
        x = 0
        for im in imgs:
            _blend_paste(canvas, im, x, 0)
            x += im.shape[1] + DIGIT_GAP
        return canvas
    return _clean_bg(np.array(generate_word(token, writer_id)))


def render_line(text: str, writer_id: int, rng=None) -> np.ndarray:
    rng = rng if rng is not None else np.random.default_rng()
    tokens = text.split()
    word_imgs = [_render_token(tok, writer_id) for tok in tokens]
    gaps = [int(rng.integers(GAP_MIN, GAP_MAX + 1)) for _ in range(max(0, len(word_imgs) - 1))]
    H = LINE_H + 2 * BASELINE_JITTER
    total_w = sum(im.shape[1] for im in word_imgs) + sum(gaps)
    canvas = np.full((H, max(1, total_w)), BG, dtype=np.uint8)
    x = 0
    for i, w in enumerate(word_imgs):
        dy = int(rng.integers(-BASELINE_JITTER, BASELINE_JITTER + 1))
        _blend_paste(canvas, w, x, BASELINE_JITTER + dy)
        x += w.shape[1] + (gaps[i] if i < len(gaps) else 0)
    return canvas

def render_block(lines: list[str], writer_id: int, seed: int | None = None):
    """Geeft (block_image, line_images) terug.

    De losse regel-afbeeldingen zijn nodig voor de OCR-evaluatie: TrOCR is een
    regel-herkenner, dus we draaien 'm per regel in plaats van op het hele blok.
    De regel-afbeeldingen zijn vrij van paper noise; die wordt alleen op het
    samengestelde blok gelegd.
    """
    rng = np.random.default_rng(seed if seed is not None else writer_id)
    line_arrs = [_clean_bg(render_line(ln, writer_id, rng=rng)) for ln in lines]
    line_gaps = [int(rng.integers(LINE_GAP_MIN, LINE_GAP_MAX + 1)) for _ in range(len(line_arrs) - 1)]
    W = max(a.shape[1] for a in line_arrs)
    H = sum(a.shape[0] for a in line_arrs) + sum(line_gaps)
    canvas = np.full((H, W), BG, dtype=np.uint8)
    y = 0
    for i, a in enumerate(line_arrs):
        _blend_paste(canvas, a, 0, y)
        y += a.shape[0] + (line_gaps[i] if i < len(line_gaps) else 0)
    canvas = _add_paper_noise(canvas, rng)
    line_imgs = [Image.fromarray(a) for a in line_arrs]
    return Image.fromarray(canvas), line_imgs


## 8 · Nederlandse test adressen

Vijf adresblokken van elk drie regels. We vermijden bewust karakters buiten de IAM vocabulary (`!"#&'()*+,-./0-9:;?A-Za-z` plus spatie).

In [ ]:
ADDRESSES = [
    ["Karol Gackowski",       "Voorhofstraat 12",     "2511 AB Den Haag"],
    ["Joep Houweling",        "Laan van Meerdervoort 81", "2517 AS Den Haag"],
    ["Jurrien de Knecht",     "Hoflaan 4 B",          "3032 AB Rotterdam"],
    ["Prime Vision BV",       "Olof Palmestraat 14",  "2616 LR Delft"],
    ["Famille van der Berg",  "Kerkstraat 27 III",    "1011 AB Amsterdam"],
]

# Drie IAM writer ids die in de paper samples redelijk verschillend lijken.
# Wissel ze gerust, geldige range is 0..338.
WRITERS = [12, 80, 201]

print(f'{len(ADDRESSES)} adressen x {len(WRITERS)} writers = {len(ADDRESSES) * len(WRITERS)} blokken om te genereren')

## 9 · Genereer alle blokken (dit is de trage cel)

DDIM met 50 timesteps per woord. Op CPU duurt het meerdere minuten per blok, op een gemiddelde GPU is het een paar seconden per woord.

In [ ]:
OUT_DIR = DIFFUSIONPEN_DIR / 'generated_dutch'
OUT_DIR.mkdir(exist_ok=True)

samples = {}  # (addr_idx, writer_id) -> PIL.Image van het hele blok
for ai, lines in enumerate(ADDRESSES):
    for w in WRITERS:
        print(f"[adr {ai+1}/{len(ADDRESSES)}] writer {w}: {' / '.join(lines)}")
        block_img, _line_imgs = render_block(lines, w)
        out_path = OUT_DIR / f'addr{ai:02d}_writer{w:03d}.png'
        block_img.save(out_path)
        samples[(ai, w)] = block_img
print('Opgeslagen naar', OUT_DIR)

## 10 · Visuele evaluatie, grid van gegenereerde blokken

Rijen zijn adressen, kolommen zijn writers. Bekijk dit op:
- letter shape consistentie binnen een rij (dezelfde writer moet als dezelfde hand aanvoelen)
- style diversiteit tussen kolommen
- omgang met cijfers, hoofdletters en spacing

In [ ]:
fig, axes = plt.subplots(len(ADDRESSES), len(WRITERS),
                         figsize=(5 * len(WRITERS), 1.8 * len(ADDRESSES)))
if len(ADDRESSES) == 1:
    axes = np.array([axes])
for ai, lines in enumerate(ADDRESSES):
    for wi, w in enumerate(WRITERS):
        ax = axes[ai, wi]
        ax.imshow(samples[(ai, w)], cmap='gray')
        ax.set_xticks([]); ax.set_yticks([])
        if ai == 0:
            ax.set_title(f'writer {w}')
        if wi == 0:
            ax.set_ylabel(f'adr {ai}', rotation=0, ha='right', va='center')
plt.tight_layout(); plt.show()

## 11 — OCR readback, CER en WER tegenover de bedoelde tekst

We gebruiken **PaddleOCR** om de gegenereerde adresblokken terug te lezen. In een aparte benchmark (zie [`ocr_vergelijking.ipynb`](ocr_vergelijking.ipynb)) hebben we vier OCR-aanpakken getoetst op 12 handmatig getranscribeerde echte Prime Vision adresblokken. PaddleOCR kwam als beste met een CER van 0.349 op echt handschrift, tegen 0.438 voor EasyOCR en boven de 0.70 voor TrOCR en de TrOCR+EasyOCR-combo.

We waren eerder van EasyOCR afgestapt omdat dat cursief schrift slecht las (een `K` werd een `y`). TrOCR bleek echter nog slechter op de rommelige echte enveloppen. PaddleOCR heeft ingebouwde regel-detectie, dus we voeren het hele blok in één keer in.

In [ ]:
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb))
        prev = cur
    return prev[-1]

def cer(ref: str, hyp: str) -> float:
    return levenshtein(ref, hyp) / max(1, len(ref))

def wer(ref: str, hyp: str) -> float:
    rs, hs = ref.split(), hyp.split()
    if not rs:
        return 1.0 if hs else 0.0
    # Levenshtein op woordniveau
    prev = list(range(len(hs) + 1))
    for i, ra in enumerate(rs, 1):
        cur = [i] + [0] * len(hs)
        for j, hb in enumerate(hs, 1):
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + (ra != hb))
        prev = cur
    return prev[-1] / len(rs)

# OCR engine: PaddleOCR.
from paddleocr import PaddleOCR

_paddle = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

def ocr(img: Image.Image) -> str:
    """OCR op een heel adresblok met PaddleOCR. Gedetecteerde regels worden van
    boven naar beneden gesorteerd en samengevoegd tot een string."""
    res = _paddle.ocr(np.array(img.convert('RGB')), cls=True)
    if not res or not res[0]:
        return ''
    items = []
    for box, (text, conf) in res[0]:
        top = min(p[1] for p in box)
        left = min(p[0] for p in box)
        items.append((top, left, text))
    items.sort()
    return ' '.join(t for _, _, t in items).strip()

ocr_engine = 'PaddleOCR'
print('OCR engine:', ocr_engine)

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

missing = [name for name in ('ADDRESSES', 'WRITERS', 'samples', 'ocr')
           if name not in globals()]
if missing:
    raise RuntimeError(
        f'Cel kan niet draaien. Eerst de eerdere cellen runnen. Ontbreekt: {missing}'
    )

results = []
if ocr is not None:
    for ai, lines in enumerate(ADDRESSES):
        ref = ' '.join(lines)
        for w in WRITERS:
            hyp = ocr(samples[(ai, w)])
            results.append({
                'addr': ai, 'writer': w,
                'reference': ref, 'ocr': hyp,
                'CER': round(cer(ref.lower(), hyp.lower()), 3),
                'WER': round(wer(ref.lower(), hyp.lower()), 3),
            })
    try:
        import pandas as pd
        df = pd.DataFrame(results)
        display(df)
        mean_cer = round(df['CER'].mean(), 3)
        mean_wer = round(df['WER'].mean(), 3)
    except ImportError:
        for r in results:
            print(r)
        mean_cer = round(np.mean([r['CER'] for r in results]), 3)
        mean_wer = round(np.mean([r['WER'] for r in results]), 3)
    print('Gemiddelde CER:', mean_cer, '| Gemiddelde WER:', mean_wer)

    # Sla resultaten op zodat de vergelijking cel ze later kan inlezen.
    out_json = DIFFUSIONPEN_DIR.parent / 'results' / 'ocr_results_diffusionpen.json'
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump({
            'model': 'DiffusionPen',
            'ocr_engine': ocr_engine,
            'mean_CER': mean_cer,
            'mean_WER': mean_wer,
            'rows': results,
        }, f, ensure_ascii=False, indent=2)
    print('Opgeslagen naar', out_json)
else:
    print('(OCR overgeslagen, geen engine beschikbaar)')

## 12 · Realisme, side-by-side met echte Prime Vision samples

Puur visueel, maar nuttig om het visuele oordeel te verankeren tegen de echte distributie waar het ons uiteindelijk om gaat.

In [ ]:
REAL_DIR = DIFFUSIONPEN_DIR.parent / 'data' / 'AddressBlock examples'
real_paths = sorted(REAL_DIR.glob('real-hw-*.png'))[:5]
print(f'Toon {len(real_paths)} echte samples vs {len(ADDRESSES)} gegenereerd (writer {WRITERS[0]})')

rows = max(len(real_paths), len(ADDRESSES))
fig, axes = plt.subplots(rows, 2, figsize=(12, 1.8 * rows))
for i in range(rows):
    if i < len(real_paths):
        axes[i, 0].imshow(Image.open(real_paths[i]).convert('L'), cmap='gray')
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    if i == 0: axes[i, 0].set_title('Echte Prime Vision sample')

    if i < len(ADDRESSES):
        axes[i, 1].imshow(samples[(i, WRITERS[0])], cmap='gray')
    axes[i, 1].set_xticks([]); axes[i, 1].set_yticks([])
    if i == 0: axes[i, 1].set_title(f'DiffusionPen, writer {WRITERS[0]}')
plt.tight_layout(); plt.show()